In [1]:
import pandas as pd 
import numpy as np 
import matplotlib as mp
from sklearn.model_selection import train_test_split

In [2]:
data = pd.read_csv('/Users/veeru/Downloads/capstone projects cs229/Semiconductor-Manufacturing-Quality-Intelligence/uci-secom.csv')

# Data EDA

In [3]:
data

,Time,0,1,2,3,4,5,6,7,8,...,581,582,583,584,585,586,587,588,589,Pass/Fail
0,2008-07-19 11:55:00,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,...,NaN,0.5005,0.0118,0.0035,2.3630,NaN,NaN,NaN,NaN,-1
1,2008-07-19 12:32:00,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,...,208.2045,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,-1
2,2008-07-19 13:17:00,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,...,82.8602,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1
3,2008-07-19 14:43:00,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,...,73.8432,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,-1
4,2008-07-19 15:22:00,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,...,NaN,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1562,2008-10-16 15:13:00,2899.41,2464.36,2179.7333,3085.3781,1.4843,100.0,82.2467,0.1248,1.3424,...,203.1720,0.4988,0.0143,0.0039,2.8669,0.0068,0.0138,0.0047,203.1720,-1
1563,2008-10-16 20:49:00,3052.31,2522.55,2198.5667,1124.6595,0.8763,100.0,98.4689,0.1205,1.4333,...,NaN,0.4975,0.0131,0.0036,2.6238,0.0068,0.0138,0.0047,203.1720,-1
1564,2008-10-17 05:26:00,2978.81,2379.78,2206.3000,1110.4967,0.8236,100.0,99.4122,0.1208,NaN,...,43.5231,0.4987,0.0153,0.0041,3.0590,0.0197,0.0086,0.0025,43.5231,-1
1565,2008-10-17 06:01:00,2894.92,2532.01,2177.0333,1183.7287,1.5726,100.0,98.7978,0.1213,1.4622,...,93.4941,0.5004,0.0178,0.0038,3.5662,0.0262,0.0245,0.0075,93.4941,-1


In [4]:
data.shape

(1567, 592)

In [5]:
data=data.drop('Time',axis=1)

## Handling the Missing Values Present in The Data

In [6]:
data.isnull().sum() # The data got a lot of missing values in each column

0             6
1             7
2            14
3            14
4            14
             ..
586           1
587           1
588           1
589           1
Pass/Fail     0
Length: 591, dtype: int64

In [7]:
print('before', data.shape) # Checking if we can remove all the rows in the missing value columns, but cannot since every row has a missing value
data2=data.dropna()
print('after', data2.shape)

before (1567, 591)
after (0, 591)


In [8]:
missing = data.isna().sum().sort_values(ascending=False)
print(missing.head(30))

157    1429
292    1429
293    1429
158    1429
492    1341
358    1341
85     1341
220    1341
246    1018
109    1018
110    1018
111    1018
518    1018
516    1018
245    1018
244    1018
382    1018
383    1018
384    1018
517    1018
578     949
579     949
580     949
581     949
346     794
73      794
72      794
345     794
112     715
247     715
dtype: int64


In [9]:
missing_perc = ((data.isna().sum()*100/1567).sort_values(ascending=False))
print(missing_perc.head(30))


157    91.193363
292    91.193363
293    91.193363
158    91.193363
492    85.577537
358    85.577537
85     85.577537
220    85.577537
246    64.964901
109    64.964901
110    64.964901
111    64.964901
518    64.964901
516    64.964901
245    64.964901
244    64.964901
382    64.964901
383    64.964901
384    64.964901
517    64.964901
578    60.561583
579    60.561583
580    60.561583
581    60.561583
346    50.670070
73     50.670070
72     50.670070
345    50.670070
112    45.628590
247    45.628590
dtype: float64


In [10]:
missing_perc.describe() # We find mean is low which confirms that percentage of missing values in a column is under 5% on avg

count    591.000000
mean       4.529871
std       15.422031
min        0.000000
25%        0.127632
50%        0.382897
75%        0.574346
max       91.193363
dtype: float64

In [11]:
bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100] # Categorising the column based on total missing values

pd.cut(
    missing_perc,
    bins=bins,
    include_lowest=True
).value_counts().sort_index()

(-0.001, 10.0]    539
(10.0, 20.0]       20
(20.0, 30.0]        0
(30.0, 40.0]        0
(40.0, 50.0]        4
(50.0, 60.0]        4
(60.0, 70.0]       16
(70.0, 80.0]        0
(80.0, 90.0]        4
(90.0, 100.0]       4
Name: count, dtype: int64

In [12]:
print(data.columns[(data.isnull().mean()*100)>80]) # These are the columns that have >80% of missing data, lets drop these column as they will affect our decision

Index(['85', '157', '158', '220', '292', '293', '358', '492'], dtype='object')


In [13]:
data=data.drop((data.columns[(data.isnull().mean()*100)>=40]),axis=1) # Deleting all columns that have more than 40% of missing values

### Implant Median Values to The Missing Column

In [14]:
under_20 = missing_perc[missing_perc <= 20].index

In [15]:
data[under_20] = data[under_20].fillna(data[under_20].median())

##

In [16]:
data.duplicated().sum()

0

In [17]:
data.isnull().sum()

0            0
1            0
2            0
3            0
4            0
            ..
586          0
587          0
588          0
589          0
Pass/Fail    0
Length: 559, dtype: int64

In [18]:
data.describe()

,0,1,2,3,4,5,6,7,8,9,...,577,582,583,584,585,586,587,588,589,Pass/Fail
count,1567.000000,1567.000000,1567.000000,1567.000000,1567.000000,1567.0,1567.000000,1567.000000,1567.000000,1567.000000,...,1567.000000,1567.000000,1567.000000,1567.000000,1567.000000,1567.000000,1567.000000,1567.000000,1567.000000,1567.000000
mean,3014.441551,2495.866110,2200.551958,1395.383474,4.171281,100.0,101.116476,0.121825,1.462860,-0.000842,...,16.642363,0.500096,0.015317,0.003846,3.067628,0.021458,0.016474,0.005283,99.652345,-0.867262
std,73.480841,80.228143,29.380973,439.837330,56.103721,0.0,6.209385,0.008936,0.073849,0.015107,...,12.485267,0.003403,0.017174,0.003719,3.576899,0.012354,0.008805,0.002866,93.864558,0.498010
min,2743.240000,2158.750000,2060.660000,0.000000,0.681500,100.0,82.131100,0.000000,1.191000,-0.053400,...,4.582000,0.477800,0.006000,0.001700,1.197500,-0.016900,0.003200,0.001000,0.000000,-1.000000
25%,2966.665000,2452.885000,2181.099950,1083.885800,1.017700,100.0,97.937800,0.121100,1.411250,-0.010800,...,11.501550,0.497900,0.011600,0.003100,2.306500,0.013450,0.010600,0.003300,44.368600,-1.000000
50%,3011.490000,2499.405000,2201.066700,1285.214400,1.316800,100.0,101.512200,0.122400,1.461600,-0.001300,...,13.817900,0.500200,0.013800,0.003600,2.757650,0.020500,0.014800,0.004600,71.900500,-1.000000
75%,3056.540000,2538.745000,2218.055500,1590.169900,1.518800,100.0,104.530000,0.123800,1.516850,0.008400,...,17.080900,0.502350,0.016500,0.004100,3.294950,0.027600,0.020300,0.006400,114.749700,-1.000000
max,3356.350000,2846.440000,2315.266700,3715.041700,1114.536600,100.0,129.252200,0.128600,1.656400,0.074900,...,96.960100,0.509800,0.476600,0.104500,99.303200,0.102800,0.079900,0.028600,737.304800,1.000000


In [19]:
data['Pass/Fail'].value_counts()

Pass/Fail
-1    1463
 1     104
Name: count, dtype: int64

In [20]:
data.corr(numeric_only=True)

,0,1,2,3,4,5,6,7,8,9,...,577,582,583,584,585,586,587,588,589,Pass/Fail
0,1.000000,-0.144161,0.004667,-0.006665,-0.010819,NaN,0.002028,0.031347,-0.052735,0.009091,...,0.008639,0.000190,0.023562,0.019994,0.023695,0.018534,-0.025867,-0.028161,0.004185,-0.025100
1,-0.144161,1.000000,0.005883,-0.008963,-0.001917,NaN,-0.025222,-0.011761,0.031244,0.024025,...,-0.010156,0.043754,0.002863,-0.001305,0.002231,-0.009430,0.017056,0.009868,0.044552,-0.002542
2,0.004667,0.005883,1.000000,0.298810,0.095881,NaN,-0.136212,-0.147116,0.023524,0.016248,...,-0.028737,-0.005971,0.015694,0.018224,0.015749,-0.025506,-0.029287,-0.030743,-0.032763,-0.000999
3,-0.006665,-0.008963,0.298810,1.000000,-0.058351,NaN,-0.685773,0.086965,-0.102839,0.066987,...,0.016888,0.008305,0.025288,0.024615,0.025882,0.035164,-0.039196,-0.033755,-0.080942,-0.024014
4,-0.010819,-0.001917,0.095881,-0.058351,1.000000,NaN,-0.074395,-0.343413,-0.025933,0.053973,...,-0.003978,0.044943,-0.001331,-0.001621,-0.001645,-0.043831,-0.031012,-0.026115,0.050792,-0.013633
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
586,0.018534,-0.009430,-0.025506,0.035164,-0.043831,NaN,-0.041299,0.058019,0.010418,0.033700,...,-0.002678,-0.016729,0.002258,0.001603,0.002745,1.000000,0.167920,0.164247,-0.486530,0.004170
587,-0.025867,0.017056,-0.029287,-0.039196,-0.031012,NaN,0.034113,-0.021518,0.022793,0.059181,...,-0.009388,-0.024507,-0.002566,-0.002442,-0.002848,0.167920,1.000000,0.974276,0.390834,0.035423
588,-0.028161,0.009868,-0.030743,-0.033755,-0.026115,NaN,0.032341,-0.020936,0.026185,0.060609,...,-0.015575,-0.020743,-0.002172,-0.001900,-0.002444,0.164247,0.974276,1.000000,0.389238,0.031207
589,0.004185,0.044552,-0.032763,-0.080942,0.050792,NaN,0.044039,-0.107904,-0.022864,0.004670,...,-0.024740,0.041449,-0.002928,-0.003246,-0.003722,-0.486530,0.390834,0.389238,1.000000,-0.002603


In [21]:
data.shape # (1567, 592) original data size

(1567, 559)

# Train, Test Splitting

In [22]:
data2 = data.copy()

In [23]:
data2 = pd.DataFrame(data2)

In [24]:
data

,0,1,2,3,4,5,6,7,8,9,...,577,582,583,584,585,586,587,588,589,Pass/Fail
0,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,0.0162,...,14.9509,0.5005,0.0118,0.0035,2.3630,0.0205,0.0148,0.0046,71.9005,-1
1,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,-0.0005,...,10.9003,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,-1
2,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,0.0041,...,9.2721,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1
3,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,-0.0124,...,8.5831,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,-1
4,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,-0.0031,...,10.9698,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1562,2899.41,2464.36,2179.7333,3085.3781,1.4843,100.0,82.2467,0.1248,1.3424,-0.0045,...,11.7256,0.4988,0.0143,0.0039,2.8669,0.0068,0.0138,0.0047,203.1720,-1
1563,3052.31,2522.55,2198.5667,1124.6595,0.8763,100.0,98.4689,0.1205,1.4333,-0.0061,...,17.8379,0.4975,0.0131,0.0036,2.6238,0.0068,0.0138,0.0047,203.1720,-1
1564,2978.81,2379.78,2206.3000,1110.4967,0.8236,100.0,99.4122,0.1208,1.4616,-0.0013,...,17.7267,0.4987,0.0153,0.0041,3.0590,0.0197,0.0086,0.0025,43.5231,-1
1565,2894.92,2532.01,2177.0333,1183.7287,1.5726,100.0,98.7978,0.1213,1.4622,-0.0072,...,19.2104,0.5004,0.0178,0.0038,3.5662,0.0262,0.0245,0.0075,93.4941,-1


In [25]:
data2

,0,1,2,3,4,5,6,7,8,9,...,577,582,583,584,585,586,587,588,589,Pass/Fail
0,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,0.0162,...,14.9509,0.5005,0.0118,0.0035,2.3630,0.0205,0.0148,0.0046,71.9005,-1
1,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,-0.0005,...,10.9003,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,-1
2,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,0.0041,...,9.2721,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1
3,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,-0.0124,...,8.5831,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,-1
4,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,-0.0031,...,10.9698,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1562,2899.41,2464.36,2179.7333,3085.3781,1.4843,100.0,82.2467,0.1248,1.3424,-0.0045,...,11.7256,0.4988,0.0143,0.0039,2.8669,0.0068,0.0138,0.0047,203.1720,-1
1563,3052.31,2522.55,2198.5667,1124.6595,0.8763,100.0,98.4689,0.1205,1.4333,-0.0061,...,17.8379,0.4975,0.0131,0.0036,2.6238,0.0068,0.0138,0.0047,203.1720,-1
1564,2978.81,2379.78,2206.3000,1110.4967,0.8236,100.0,99.4122,0.1208,1.4616,-0.0013,...,17.7267,0.4987,0.0153,0.0041,3.0590,0.0197,0.0086,0.0025,43.5231,-1
1565,2894.92,2532.01,2177.0333,1183.7287,1.5726,100.0,98.7978,0.1213,1.4622,-0.0072,...,19.2104,0.5004,0.0178,0.0038,3.5662,0.0262,0.0245,0.0075,93.4941,-1


In [26]:
for i in range(len(data)):
    if data2.loc[i, "Pass/Fail"] == -1:
        data2.loc[i, "Pass/Fail"] = 0
    elif data2.loc[i, "Pass/Fail"] == 1:
        data2.loc[i, "Pass/Fail"] = 1

In [27]:
X=data2.drop('Pass/Fail',axis=1)
y=data2['Pass/Fail']

In [28]:
X_train, X_test, y_train, y_test= train_test_split(
    X, y, test_size=0.30, random_state=42
)

In [54]:
y_train.value_counts()

Pass/Fail
0    1023
1      73
Name: count, dtype: int64

# Performing Logistic Regression 

In [29]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [30]:
# Create scaler
scaler = StandardScaler()

# Fit ONLY on training data
X_train = scaler.fit_transform(X_train)

# Use the same scaler on test data
X_test = scaler.transform(X_test)

In [31]:
X_train = np.c_[np.ones(X_train.shape[0]), X_train]
X_test = np.c_[np.ones(X_test.shape[0]), X_test]

In [32]:
def signmoid(a):
    return 1/(1+np.exp(-a))

In [33]:
def logi_regression(X,y):
    m,n = X.shape
    theta = np.zeros(n)
    theta2 = np.zeros(n)
    i=0
    while True:
        i+=1
        check = (X.T @ (signmoid(X@theta) - y))+100*theta #Adding regularization
        theta2 = theta - 0.001/m*check
        z = X @ theta
        if (i%1000)==0:
            print('Iteration ', i)
            print(np.linalg.norm(theta))
            print("max |z|:", np.max(np.abs(z)))
        if np.linalg.norm(theta2 - theta) < 1e-6:
            theta = theta2
            break
        theta=theta2
    return theta

In [34]:
val = logi_regression(X_train,y_train)
val

Iteration  1000
0.3869791533500364
max |z|: 1.4077704694632163
Iteration  2000
0.658973655693486
max |z|: 1.5383598599138517
Iteration  3000
0.8585990201023104
max |z|: 1.7873064952926465
Iteration  4000
1.008014270241426
max |z|: 2.024029389796247
Iteration  5000
1.121565523912162
max |z|: 2.234772955879918
Iteration  6000
1.2089158329997813
max |z|: 2.402875574935657
Iteration  7000
1.2767683615207739
max |z|: 2.538688372262044
Iteration  8000
1.3298919659326083
max |z|: 2.6490865273611077
Iteration  9000
1.3717528824113125
max |z|: 2.7390411999985784
Iteration  10000
1.4049160497367266
max |z|: 2.8123519353467272
Iteration  11000
1.4313077961042662
max |z|: 2.8720271196047373
Iteration  12000
1.452392621696003
max |z|: 2.920501108942368
Iteration  13000
1.469295193626788
max |z|: 2.9597691564918565
Iteration  14000
1.4828864271650994
max |z|: 2.991478028735359
Iteration  15000
1.493845419406976
max |z|: 3.0169911585514493
Iteration  16000
1.5027047665528848
max |z|: 3.03743818109323

array([-1.46361774e+00, -4.50400342e-02, -1.73033312e-02, -5.86518024e-03,
       -6.47350072e-03, -4.36235128e-03,  0.00000000e+00, -2.78801212e-02,
        8.45560175e-03, -1.07155544e-04, -5.53141046e-03, -5.30083952e-03,
       -5.37692468e-02,  3.97813843e-03,  0.00000000e+00, -3.10144074e-02,
        1.00670812e-02,  1.47388258e-02, -2.73087160e-03, -1.01225117e-02,
        8.34694119e-03,  1.99217936e-02,  3.59314008e-02, -4.09593728e-02,
        9.11426193e-03,  4.64017189e-03,  1.07746256e-02, -3.23180971e-02,
        2.51167555e-02, -6.53391191e-03,  7.48237437e-03, -1.92839132e-02,
       -2.76201092e-02,  4.02298076e-02,  3.91715899e-02, -1.46799112e-02,
       -1.30187230e-02,  1.46803754e-02,  2.76455966e-02, -4.06006643e-02,
       -1.53411791e-02, -4.27893233e-02, -2.07319928e-02,  0.00000000e+00,
        1.11425157e-02, -9.49475230e-03,  8.68553474e-03,  1.35161081e-02,
        8.19599227e-03, -5.36274846e-03,  0.00000000e+00,  1.32457812e-02,
       -2.96959941e-02,  

In [58]:
predictions = 1/(1+np.exp(-X_train @ val))
binary_predictions = (predictions >= 0.5).astype(int)
accuracy = np.mean(binary_predictions == y_train)
print(f"Accuracy on training set: {accuracy * 100:.2f}%")

Accuracy on training set: 94.53%


In [75]:
predictions = 1/(1+np.exp(-X_test @ val))
binary_predictions = (predictions >= 0.5).astype(int)
accuracy = np.mean(binary_predictions == y_test)
print(f"Accuracy on test set: {accuracy * 100:.2f}%")

Accuracy on test set: 92.78%


In [74]:
# Confusion matrix
TP = np.sum((y_test == 1) & (binary_predictions == 1))
TN = np.sum((y_test == 0) & (binary_predictions == 0))
FP = np.sum((y_test == 0) & (binary_predictions == 1))
FN = np.sum((y_test == 1) & (binary_predictions == 0))

print("Confusion Matrix:")
print(np.array([
    [TN, FP],
    [FN, TP]
]))

print('Precision :',TP/(TP+FP))
print('Recall :', TP/(TP+FN))

Confusion Matrix:
[[393  47]
 [ 21  10]]
Precision : 0.17543859649122806
Recall : 0.3225806451612903


#### The Value never converges and hence L2 regularisation was added to the logistic Regression

#### Higher Learning Rate Not Converging as the value keeps missing the convergence point and hence it continues to go back and forth

#### Recall of 71% means of all the failed chips out model predicted correctly 71% times, whereas only 10.6% of the total chips predicted to have failed actually failed -> this means the model predicting chips fail very often and misclassifies

#### 1. Recall = "Did I catch the failures?", Imagine there are 100 actual failed wafers. Your model catches 80 of them.

#### 2. Precision = "When I say failure, am I usually right?", Now imagine your model flags 200 wafers as failures. But only 80 of those are actually failures.


# Newton's Method to Check If Convergence

In [194]:
def Newton(X,y):
    m,n = X.shape
    theta = np.zeros(n)
    theta2 = np.zeros(n)
    def sigmoid(a):
        return 1/(1+np.exp(-a))
    while True:
        j = 1/m * X.T @ (sigmoid((X @ theta)) - y)
        s_diagonal = (sigmoid((X @ theta)) * (1- sigmoid((X @ theta))))[:, np.newaxis]
        h = 1/m * X.T @ ( s_diagonal * X)
        theta2 = theta - np.linalg.solve(h, j)
        if np.linalg.norm(theta2 - theta) < 1e-5:
            theta = theta2
            break
        theta = theta2




In [195]:
val = Newton(X_train,y_train)

LinAlgError: Singular matrix

#### Newtons Method was explored to be used however it faces issue as matrix is not invertible (Singular Matrix)

# Naive Bayes 

In [212]:
data

,0,1,2,3,4,5,6,7,8,9,...,577,582,583,584,585,586,587,588,589,Pass/Fail
0,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,0.0162,...,14.9509,0.5005,0.0118,0.0035,2.3630,0.0205,0.0148,0.0046,71.9005,0
1,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,-0.0005,...,10.9003,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,0
2,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,0.0041,...,9.2721,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1
3,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,-0.0124,...,8.5831,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,0
4,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,-0.0031,...,10.9698,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1562,2899.41,2464.36,2179.7333,3085.3781,1.4843,100.0,82.2467,0.1248,1.3424,-0.0045,...,11.7256,0.4988,0.0143,0.0039,2.8669,0.0068,0.0138,0.0047,203.1720,0
1563,3052.31,2522.55,2198.5667,1124.6595,0.8763,100.0,98.4689,0.1205,1.4333,-0.0061,...,17.8379,0.4975,0.0131,0.0036,2.6238,0.0068,0.0138,0.0047,203.1720,0
1564,2978.81,2379.78,2206.3000,1110.4967,0.8236,100.0,99.4122,0.1208,1.4616,-0.0013,...,17.7267,0.4987,0.0153,0.0041,3.0590,0.0197,0.0086,0.0025,43.5231,0
1565,2894.92,2532.01,2177.0333,1183.7287,1.5726,100.0,98.7978,0.1213,1.4622,-0.0072,...,19.2104,0.5004,0.0178,0.0038,3.5662,0.0262,0.0245,0.0075,93.4941,0


#### Note: Pass -> 0 ; Fail -> 1

In [213]:
for i in range(len(data)):
    if data.loc[i, "Pass/Fail"] == -1:
        data.loc[i, "Pass/Fail"] = 0
    elif data.loc[i, "Pass/Fail"] == 1:
        data.loc[i, "Pass/Fail"] = 1

####

In [214]:
train_data, test_data = train_test_split(
    data,
    test_size=0.30,
    random_state=42
)

In [199]:
data['Pass/Fail'].value_counts()

Pass/Fail
0    1463
1     104
Name: count, dtype: int64

In [215]:
positive_train = train_data[train_data['Pass/Fail']==1]
positive_train = positive_train.drop('Pass/Fail',axis=1)

In [216]:
negative_train = train_data[train_data['Pass/Fail']==0]
negative_train = negative_train.drop('Pass/Fail',axis=1)

In [217]:
prob_positive = len(positive_train)/(len(positive_train)+len(negative_train))
prob_negative = len(negative_train)/(len(positive_train)+len(negative_train))

In [218]:
mean_positive = np.mean(positive_train, axis = 0)
std_positive = np.std(positive_train, axis = 0)

In [219]:
mean_negative = np.mean(negative_train, axis = 0)
std_negative = np.std(negative_train, axis = 0)

In [220]:
X_test = test_data.drop('Pass/Fail', axis=1)
Y_test = test_data['Pass/Fail']

In [221]:
likelihood_positive = (1/(np.sqrt(2*(22/7))*std_positive))*np.exp(-(X_test - mean_positive)**2/(2*std_positive**2))

In [222]:
likelihood_negative = (1/(np.sqrt(2*(22/7))*std_negative))*np.exp(-(X_test - mean_negative)**2/(2*std_negative**2))

In [223]:
probability_positive = np.log(prob_positive) + np.sum(np.log(likelihood_positive),axis=1)
probability_negative = np.log(prob_negative) + np.sum(np.log(likelihood_negative), axis=1)

/Users/veeru/anaconda3/lib/python3.11/site-packages/pandas/core/internals/blocks.py:393: RuntimeWarning: divide by zero encountered in log
  result = func(self.values, **kwargs)
/Users/veeru/anaconda3/lib/python3.11/site-packages/pandas/core/internals/blocks.py:393: RuntimeWarning: divide by zero encountered in log
  result = func(self.values, **kwargs)


In [224]:
predictions = []

for i in range(len(X_test)):

    if probability_positive.iloc[i] >= probability_negative.iloc[i]:
        predictions.append(1)
    else:
        predictions.append(0)

In [225]:
predictions = np.array(predictions)

accuracy = np.mean(predictions == Y_test.to_numpy())

print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 20.38%


In [226]:
# Confusion matrix
TP = np.sum((Y_test == 1) & (predictions == 1))
TN = np.sum((Y_test == 0) & (predictions == 0))
FP = np.sum((Y_test == 0) & (predictions == 1))
FN = np.sum((Y_test == 1) & (predictions == 0))

print("Confusion Matrix:")
print(np.array([
    [TN, FP],
    [FN, TP]
]))

print('Precision :',TP/(TP+FP))
print('Recall :', TP/(TP+FN))

Confusion Matrix:
[[ 72 368]
 [  7  24]]
Precision : 0.061224489795918366
Recall : 0.7741935483870968


##### The Recall Value here is 77.4% which is more than 9% Recall we get from Logistic Regression. But overall accuracy very poor and this model has a very bad precision as well

# Performing SVM operation on this dataset

In [126]:
data

,0,1,2,3,4,5,6,7,8,9,...,577,582,583,584,585,586,587,588,589,Pass/Fail
0,3030.93,2564.00,2187.7333,1411.1265,1.3602,100.0,97.6133,0.1242,1.5005,0.0162,...,14.9509,0.5005,0.0118,0.0035,2.3630,0.0205,0.0148,0.0046,71.9005,0
1,3095.78,2465.14,2230.4222,1463.6606,0.8294,100.0,102.3433,0.1247,1.4966,-0.0005,...,10.9003,0.5019,0.0223,0.0055,4.4447,0.0096,0.0201,0.0060,208.2045,0
2,2932.61,2559.94,2186.4111,1698.0172,1.5102,100.0,95.4878,0.1241,1.4436,0.0041,...,9.2721,0.4958,0.0157,0.0039,3.1745,0.0584,0.0484,0.0148,82.8602,1
3,2988.72,2479.90,2199.0333,909.7926,1.3204,100.0,104.2367,0.1217,1.4882,-0.0124,...,8.5831,0.4990,0.0103,0.0025,2.0544,0.0202,0.0149,0.0044,73.8432,0
4,3032.24,2502.87,2233.3667,1326.5200,1.5334,100.0,100.3967,0.1235,1.5031,-0.0031,...,10.9698,0.4800,0.4766,0.1045,99.3032,0.0202,0.0149,0.0044,73.8432,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1562,2899.41,2464.36,2179.7333,3085.3781,1.4843,100.0,82.2467,0.1248,1.3424,-0.0045,...,11.7256,0.4988,0.0143,0.0039,2.8669,0.0068,0.0138,0.0047,203.1720,0
1563,3052.31,2522.55,2198.5667,1124.6595,0.8763,100.0,98.4689,0.1205,1.4333,-0.0061,...,17.8379,0.4975,0.0131,0.0036,2.6238,0.0068,0.0138,0.0047,203.1720,0
1564,2978.81,2379.78,2206.3000,1110.4967,0.8236,100.0,99.4122,0.1208,1.4616,-0.0013,...,17.7267,0.4987,0.0153,0.0041,3.0590,0.0197,0.0086,0.0025,43.5231,0
1565,2894.92,2532.01,2177.0333,1183.7287,1.5726,100.0,98.7978,0.1213,1.4622,-0.0072,...,19.2104,0.5004,0.0178,0.0038,3.5662,0.0262,0.0245,0.0075,93.4941,0


In [127]:
for i in range(len(data)):
    if data.loc[i, "Pass/Fail"] == 0:
        data.loc[i, "Pass/Fail"] = -1
    elif data.loc[i, "Pass/Fail"] == 1:
        data.loc[i, "Pass/Fail"] = 1

In [128]:
data.shape[0]

1567

In [129]:
X=data.drop('Pass/Fail',axis=1)
y=data['Pass/Fail']

In [130]:
X_train, X_test, y_train, y_test= train_test_split(
    X, y, test_size=0.30, random_state=42
)

In [145]:
epoch=300
c=1
w=np.zeros(X.shape[1])
b=0
for i in range(epoch):
    dw=np.zeros(X_train.shape[1])
    db=0
    for k in range(X_train.shape[0]):
        x=X_train.iloc[k]
        y=y_train.iloc[k]
        score=np.dot(w,x)+b
        if y*score<1:
            dw+=-c*y*x
            db+=-c*y
    dw+=w
    dw=dw/X.shape[0]
    db=db/X.shape[0]

    w=w-0.01*dw
    b=b-0.01*db
    


        

In [146]:
predictions = []

for k in range(X_test.shape[0]):

    x = X_test.iloc[k]

    score = np.dot(w, x) + b

    if score >= 0:
        prediction = 1
    else:
        prediction = -1

    predictions.append(prediction)

correct = 0

for k in range(len(y_test)):

    if predictions[k] == y_test.iloc[k]:
        correct += 1

accuracy = correct / len(y_test)

print("Accuracy:", accuracy)
print("Accuracy %:", accuracy * 100)

Accuracy: 0.9341825902335457
Accuracy %: 93.41825902335457


In [147]:
TP = 0
TN = 0
FP = 0
FN = 0

for k in range(len(y_test)):

    if y_test.iloc[k] == 1 and predictions[k] == 1:
        TP += 1

    elif y_test.iloc[k] == -1 and predictions[k] == -1:
        TN += 1

    elif y_test.iloc[k] == -1 and predictions[k] == 1:
        FP += 1

    elif y_test.iloc[k] == 1 and predictions[k] == -1:
        FN += 1

print("Confusion Matrix")
print("                 Predicted")
print("                 -1     +1")
print("Actual -1      ", TN, "   ", FP)
print("Actual +1      ", FN, "   ", TP)

Confusion Matrix
                 Predicted
                 -1     +1
Actual -1       440     0
Actual +1       31     0


#### The Precision and Recall is 0 which mean sthe model predicts everything to be -1 or no failure, this is not somethign we want for the industry and therefore SVM is not a good model.